# 04 · Modelling — Favorita

Modellvergleich auf dem gleichen chronologischen Split:

| Gruppe | Modell | Datenformat |
|---|---|---|
| Statistisch | SARIMAX | 1D-Serie pro Store |
| Statistisch | Prophet | 1D-Serie + exogene Regressoren |
| ML | XGBoost | 2D-Feature-Matrix |
| ML | LightGBM | 2D-Feature-Matrix |
| Deep Learning | PatchTST | Sequenzen via neuralforecast |
| Deep Learning | NHITS | Sequenzen + stat/hist exog via neuralforecast |

## 0 · Imports & Setup

In [1]:
import os
import sys
import pathlib
import tempfile
from pathlib import Path

os.environ['CMDSTANPY_NOT_USE_POLARS'] = 'True'  # muss vor prophet-Import stehen

sys.path.append(os.path.abspath('../03_src'))

import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.models import NHITS


from utilis import run_prophet, run_sarimax, evaluate, run_xgb, run_lgbm, to_nf_format
from config import (FINAL, TARGET_COL, FEATURE_COLS,EXOG_COLS, HIST_EXOG, STAT_EXOG,TRAIN_END, VAL_END, LOOKBACK, HORIZON)

RESULTS = FINAL / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
print('Setup abgeschlossen ✓')

c:\Users\maxkr\E-Commerce-Time-Series-Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-01 10:23:13,048	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-05-01 10:23:13,632	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Importing plotly failed. Interactive plots will not work.


Setup abgeschlossen ✓


## 1 · Daten laden & Split

In [2]:
df = pl.read_parquet(FINAL / 'final_dataset.parquet')

train = df.filter(pl.col('date') <= TRAIN_END)
val   = df.filter((pl.col('date') > TRAIN_END) & (pl.col('date') <= VAL_END))
test  = df.filter(pl.col('date') > VAL_END)

print(f'Train: {train.shape}  {train["date"].min()} → {train["date"].max()}')
print(f'Val:   {val.shape}    {val["date"].min()}   → {val["date"].max()}')
print(f'Test:  {test.shape}   {test["date"].min()}  → {test["date"].max()}')

stores = train['store_nbr'].unique().sort().to_list()
print(f'Stores: {len(stores)}')

Train: (70114, 37)  2013-01-29 → 2016-12-31
Val:   (7965, 37)    2017-01-01   → 2017-05-31
Test:  (4104, 37)   2017-06-01  → 2017-08-15
Stores: 53


In [3]:
# Format 1: ML-Feature-Matrizen (für XGBoost & LightGBM)
X_train = train.select(FEATURE_COLS).to_numpy()
y_train = train.select(TARGET_COL).to_numpy().ravel()

X_val   = val.select(FEATURE_COLS).to_numpy()
y_val   = val.select(TARGET_COL).to_numpy().ravel()

X_test  = test.select(FEATURE_COLS).to_numpy()
y_test  = test.select(TARGET_COL).to_numpy().ravel()

print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}  |  X_test: {X_test.shape}')
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

X_train: (70114, 33)  |  X_val: (7965, 33)  |  X_test: (4104, 33)
Features (33): ['store_type_enc', 'cluster', 'month', 'weekday', 'quarter', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'year', 'lag_1', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'rolling_mean_7_days', 'rolling_mean_14_days', 'rolling_mean_28_days', 'wow_growth', 'mom_growth', 'same_weekday_last_week', 'same_weekday_4weeks_ago', 'sales_vs_week_avg', 'wow_vs_month_avg', 'diff_1', 'diff_7', 'diff_28', 'oil_price', 'oil_price_ma7', 'is_national_holiday', 'is_day_before_holiday', 'is_day_after_holiday', 'is_holiday_window']


In [5]:
train_nf = to_nf_format(train)
val_nf   = to_nf_format(val)

# static_df: ein Eintrag pro Store (für NHITS)
static_df = (
    train.select(['store_nbr'] + STAT_EXOG)
         .unique(subset=['store_nbr'])
         .sort('store_nbr')
         .with_columns(pl.col('store_nbr').cast(pl.Utf8).alias('unique_id'))
         .select(['unique_id'] + STAT_EXOG)
         .with_columns([pl.col(c).cast(pl.Float32) for c in STAT_EXOG])
         .to_pandas()
)

print('NeuralForecast Format:')
print(train_nf.dtypes)
print(f'static_df shape: {static_df.shape}')
train_nf.head(3)

NeuralForecast Format:
unique_id                        object
ds                       datetime64[ms]
y                                 int64
store_type_enc                  float32
cluster                         float32
oil_price                       float32
is_national_holiday             float32
is_day_before_holiday           float32
is_day_after_holiday            float32
dtype: object
static_df shape: (53, 3)


,unique_id,ds,y,store_type_enc,cluster,oil_price,is_national_holiday,is_day_before_holiday,is_day_after_holiday
0,1,2013-01-30,1877,2.0,13.0,97.980003,0.0,0.0,0.0
1,1,2013-01-31,1707,2.0,13.0,97.650002,0.0,0.0,0.0
2,1,2013-02-01,1806,2.0,13.0,97.459999,0.0,0.0,0.0


## 2 · Statistisch: SARIMAX

Läuft pro Store separat. Exogene Regressoren: Ölpreis + Feiertags-Flags.  
Saisonalität: `seasonal_order=(1,1,1,7)` → wöchentlich.

In [ ]:
preds_sarimax = run_sarimax(
    train, val,
    stores=stores,
    exog_cols=EXOG_COLS
)
preds_sarimax.write_parquet(RESULTS / 'val_sarimax.parquet')
preds_sarimax.head(5)

## 3 · Statistisch: Prophet

Läuft pro Store separat. Exogene Regressoren: Ölpreis.

In [ ]:
preds_prophet = run_prophet(
    train, val,
    stores=stores,
    extra_regressors=['oil_price']
)
preds_prophet.write_parquet(RESULTS / 'val_prophet.parquet')
preds_prophet.head(5)

## 4 · ML: XGBoost & LightGBM

Globale Modelle auf der gesamten Feature-Matrix. Kein Store-Loop nötig.

In [9]:
model_xgb, metrics_xgb = run_xgb(X_train, y_train, X_val, y_val)

preds_xgb = (
    val.select(['store_nbr', 'date', TARGET_COL])
    .rename({TARGET_COL: 'y_true'})
    .with_columns(
        pl.Series('pred_xgb', model_xgb.predict(X_val).astype(float))
    )
)
preds_xgb.write_parquet(RESULTS / 'val_xgb.parquet')

# Modell für Feature Importance speichern
import joblib
joblib.dump(model_xgb, RESULTS / 'model_xgb.pkl')
print('XGBoost ✓')
preds_xgb.head(5)

XGBoost Val           MAE=11.5  RMSE=24.8  MAPE=0.7%
XGBoost ✓


store_nbr,date,y_true,pred_xgb
i64,date,i64,f64
1,2017-01-02,516,477.053864
1,2017-01-03,1946,1892.806641
1,2017-01-04,1905,1902.989868
1,2017-01-05,1807,1820.155273
1,2017-01-06,1856,1866.127197


In [10]:
model_lgbm, metrics_lgbm = run_lgbm(X_train, y_train, X_val, y_val)

preds_lgbm = (
    val.select(['store_nbr', 'date', TARGET_COL])
    .rename({TARGET_COL: 'y_true'})
    .with_columns(
        pl.Series('pred_lgbm', model_lgbm.predict(X_val).astype(float))
    )
)
preds_lgbm.write_parquet(RESULTS / 'val_lgbm.parquet')

joblib.dump(model_lgbm, RESULTS / 'model_lgbm.pkl')
print('LightGBM ✓')
preds_lgbm.head(5)

LightGBM Val          MAE=13.0  RMSE=26.4  MAPE=0.9%
LightGBM ✓


store_nbr,date,y_true,pred_lgbm
i64,date,i64,f64
1,2017-01-02,516,352.965693
1,2017-01-03,1946,1896.734631
1,2017-01-04,1905,1918.022137
1,2017-01-05,1807,1819.045863
1,2017-01-06,1856,1860.779171


## 5 · Deep Learning: PatchTST

Transformer-basiertes Modell auf Patch-Ebene.  
Keine exogenen Features (PatchTST unterstützt keine stat/hist exog).

In [6]:
import torch
torch.set_num_threads(1)

import torch.multiprocessing
torch.multiprocessing.set_start_method('spawn', force=True)

In [7]:
import torch
torch.set_num_threads(1)

from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST

model_patchtst = PatchTST(
    h=HORIZON,
    input_size=LOOKBACK,
    patch_len=16,
    stride=8,
    encoder_layers=2,
    n_heads=8,
    hidden_size=64,
    linear_hidden_size=128,
    dropout=0.2,
    fc_dropout=0.2,
    scaler_type='standard',
    max_steps=200,
    batch_size=64,
    learning_rate=1e-4,
    early_stop_patience_steps=10,
    val_check_steps=25,
    random_seed=42,
)

nf_patchtst = NeuralForecast(models=[model_patchtst], freq='D')
nf_patchtst.fit(
    df=train_nf[['unique_id', 'ds', 'y']],
    val_size=len(val['date'].unique())
)

preds_patchtst = nf_patchtst.predict()

preds_patchtst_pl = (
    pl.from_pandas(preds_patchtst.reset_index())
    .rename({'unique_id': 'store_nbr', 'ds': 'date', 'PatchTST': 'pred_patchtst'})
    .with_columns([pl.col('store_nbr').cast(pl.Int64), pl.col('date').cast(pl.Date)])
    .join(val.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'}),
          on=['store_nbr', 'date'], how='left')
)
preds_patchtst_pl.write_parquet(RESULTS / 'val_patchtst.parquet')
print('PatchTST ✓')
preds_patchtst_pl.head(5)

Seed set to 42


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name         | Type              | Params | Mode  | FLOPs
-------------------------------------------------------------------
0 | loss         | MAE               | 0      | train | 0    
1 | padder_train | ConstantPad1d     | 0      | train | 0    
2 | scaler       | TemporalNorm      | 0      | train | 0    
3 | model        | PatchTST_backbone | 151 K  | train | 0    
-------------------------------------------------------------------
151 K     Trainable params
2         Non-trainable params
151 K     Total params
0.606     Total estimated model params size (MB)
65        Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 199: 100%|██████████| 1/1 [00:08<00:00,  0.12it/s, v_num=27, train_loss_step=1.480, train_loss_epoch=1.480, valid_loss=157.0]

`Trainer.fit` stopped: `max_steps=200` reached.


Epoch 199: 100%|██████████| 1/1 [00:08<00:00,  0.12it/s, v_num=27, train_loss_step=1.480, train_loss_epoch=1.480, valid_loss=157.0]

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 17.08it/s]
PatchTST ✓


index,store_nbr,date,pred_patchtst,y_true
i64,i64,date,f32,i64
0,1,2017-01-01,1199.482666,null
1,1,2017-01-02,509.190308,516
2,1,2017-01-03,1389.983765,1946
3,1,2017-01-04,1367.234863,1905
4,1,2017-01-05,1266.092529,1807


## 6 · Deep Learning: NHITS

Hierarchisches Interpolations-Modell.  
Unterstützt statische (`store_type_enc`, `cluster`) und historische exogene Features (`oil_price`, Feiertage).  
Deutlich schneller als TFT auf CPU bei vergleichbarer Qualität.

In [8]:
from neuralforecast.models import NHITS

required_cols_nhits = ['unique_id', 'ds', 'y'] + HIST_EXOG

model_nhits = NHITS(
    h=HORIZON,
    input_size=LOOKBACK,
    stat_exog_list=STAT_EXOG,
    hist_exog_list=HIST_EXOG,
    scaler_type='standard',
    max_steps=200,
    batch_size=128,
    learning_rate=1e-3,
    early_stop_patience_steps=5,
    val_check_steps=10,
    random_seed=42,
)

nf_nhits = NeuralForecast(models=[model_nhits], freq='D')
nf_nhits.fit(
    df=train_nf[required_cols_nhits],
    static_df=static_df,
    val_size=len(val['date'].unique())
)

preds_nhits = nf_nhits.predict()

preds_nhits_pl = (
    pl.from_pandas(preds_nhits.reset_index())
    .rename({'unique_id': 'store_nbr', 'ds': 'date', 'NHITS': 'pred_nhits'})
    .with_columns([pl.col('store_nbr').cast(pl.Int64), pl.col('date').cast(pl.Date)])
    .join(val.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'}),
          on=['store_nbr', 'date'], how='left')
)
preds_nhits_pl.write_parquet(RESULTS / 'val_nhits.parquet')
print('NHITS ✓')
preds_nhits_pl.head(5)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name         | Type          | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | loss         | MAE           | 0      | train | 0    
1 | padder_train | ConstantPad1d | 0      | train | 0    
2 | scaler       | TemporalNorm  | 0      | train | 0    
3 | blocks       | ModuleList    | 4.6 M  | train | 0    
---------------------------------------------------------------
4.6 M     Trainable params
0         Non-trainable params
4.6 M     Total params
18.530    Total estimated model params size (MB)
34        Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 199: 100%|██████████| 1/1 [00:02<00:00,  0.36it/s, v_num=29, train_loss_step=0.470, train_loss_epoch=0.470, valid_loss=155.0]    

`Trainer.fit` stopped: `max_steps=200` reached.


Epoch 199: 100%|██████████| 1/1 [00:02<00:00,  0.36it/s, v_num=29, train_loss_step=0.470, train_loss_epoch=0.470, valid_loss=155.0]

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 39.11it/s]
NHITS ✓


index,store_nbr,date,pred_nhits,y_true
i64,i64,date,f32,i64
0,1,2017-01-01,921.227051,null
1,1,2017-01-02,645.114929,516
2,1,2017-01-03,1058.248047,1946
3,1,2017-01-04,669.589355,1905
4,1,2017-01-05,1039.077515,1807
